In [15]:
from datasets import load_dataset

dataset = load_dataset("MegaScience/TextbookReasoning")

print(dataset)
print("\nColumn names:", dataset["train"].column_names)
print("Number of examples:", len(dataset["train"]))

Generating train split: 100%|██████████| 651840/651840 [00:01<00:00, 399852.80 examples/s]


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'subject', 'reference_answer'],
        num_rows: 651840
    })
})

Column names: ['question', 'answer', 'subject', 'reference_answer']
Number of examples: 651840


In [19]:
print(dataset["train"][150000])

{'question': 'Evaluate the integral \\( \\int_{-\\infty}^{+\\infty} \\frac{x \\, dx}{x^4 + 1} \\).', 'answer': 'The integrand \\( \\frac{x}{x^4 + 1} \\) is an odd function because \\( \\frac{-x}{(-x)^4 + 1} = -\\frac{x}{x^4 + 1} \\). For any odd function \\( f(x) \\), the integral over symmetric limits \\([-a, a]\\) is zero:\n\\[\n\\int_{-a}^{a} f(x) \\, dx = 0.\n\\]\nExtending this to infinite limits, the integral remains zero:\n\\[\n\\int_{-\\infty}^{+\\infty} \\frac{x \\, dx}{x^4 + 1} = \\boxed{0}.\n\\]', 'subject': 'math', 'reference_answer': '0'}


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "TinyLlama/TinyLlama_v1.1"  # base version

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,   
    device_map="auto"        
)

print("Model loaded!")
print(f"Device: {next(model.parameters()).device}")

/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:05<00:00, 34.63it/s]
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.10/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/Users/dipeshpaneru/Documents/UniveristyWork/Adv

Model loaded!
Device: mps:0


In [5]:
def ask(question, max_new_tokens=200):
    """
    Send a question to the base model and print the response.
    The base model has NO instruction formatting — we just feed
    raw text and see what it generates next.
    """
    inputs = tokenizer(question, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=5,  # stops it repeating itself
        )

    # Decode only the NEW tokens (not the input question)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print(f"QUESTION:\n{question}")
    print(f"\nMODEL OUTPUT:\n{response}")
    print("\n" + "="*60 + "\n")

In [7]:
# Test with questions typical of an advanced AI course
# Watch how the base model responds — does it answer? Does it ramble?
# Does it even understand it's being asked a question?

questions = [
    "What is backpropagation?",

    "Explain the difference between supervised and unsupervised learning.",

    "What is the vanishing gradient problem and how do we solve it?",

    "Question: What does the attention mechanism do in a transformer?\nAnswer:",

    "In the country of Nepal, who was madan bhandari"
]

for q in questions:
    ask(q)

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is backpropagation?

MODEL OUTPUT:

What are the different types of backpropagating errors?
How does backpropagated error work with neural networks and why it is not a good idea to use this approach for training deep learning models in general.
Back propagate through time:
When you have an NN, there will be many parameters, weights (variables), biases (values). For each layer, we can compute derivatives and compute gradients as well. Gradient descent is based on these calculations. The derivative is calculated using first order Taylor series approximation which gives us the gradient with respect to input variables or weights of layers one by one. We then calculate partial derivative at that point i.e. gradient with respect to weight w0 = ∂y/∂x1, y=f(w) where f(. . . ,.) denotes activation function. This gives us gradient w0 − tΔw0
where ∆w stands for change from previous step. At next iteration, we




Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the difference between supervised and unsupervised learning.

MODEL OUTPUT:
The unsupervised learning is much more flexible, it can learn about the data with no human intervention whereas in case of supervised learning you need to label each sample individually which is a tedious task for many businesses.
In supervised learning, the input features are classified into several classes by the model while in unsupervised learning these labels cannot be given because there will not any information on how to identify samples from one group or another. Supervised learning is used when we want to create models that generalize well across different datasets as opposed to a single dataset. In this type of machine learning models are trained using only labeled examples.
Unsupervised Learning is where we do not give the classification labels but instead give some inputs like clustering them together so they become easier to find later on down the line. This technique can be appli

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is the vanishing gradient problem and how do we solve it?

MODEL OUTPUT:

How to solve all kinds of problems with mathematics, statistics and probability.




Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Question: What does the attention mechanism do in a transformer?
Answer:

MODEL OUTPUT:
This is how I would implement it.

A: The answer to your question depends on what you want from your input image, and if that's what we are talking about at all or not.
For example if my input was an image of a person with their face covered by something then i might use that as opposed to a face for the same reason someone might take images of a cat without its tail but have no problem using them instead (because they are still cats).  So if we were going through a bunch of images we could just select one based off our own preference rather than trying to guess which part of each image most closely resembles the given object so long as there isn't too much overlap between the two (otherwise the model will learn nothing useful) - this can be achieved via learning weighted thresholds depending upon whether some regions should be more important/desired compared others where different regions

In [ ]:
# Run the model on every test question and store outputs
# Run this cell at each stage (base, fine-tuned, +RAG)
# and save the results separately each time



In [ ]:
import torch
import numpy as np

def compute_perplexity(model, tokenizer, texts):
    """
    Lower perplexity = model is more confident and coherent.
    A base model will have HIGH perplexity on Q&A style text.
    A fine-tuned model will have MUCH LOWER perplexity.
    This is your clearest numeric signal of improvement.
    """
    model.eval()
    perplexities = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        perplexities.append(torch.exp(loss).item())

    return {
        "mean_perplexity":   round(np.mean(perplexities), 2),
        "median_perplexity": round(np.median(perplexities), 2),
    }

# Compute on the reference answers — asking "how likely is the model
# to produce text like these good answers?"
reference_texts = [item["reference"] for item in test_set]
perp_scores = compute_perplexity(model, tokenizer, reference_texts)

print("PERPLEXITY SCORES (base model)")
print(f"  Mean:   {perp_scores['mean_perplexity']}")
print(f"  Median: {perp_scores['median_perplexity']}")
print("\n  (lower is better — expect this to drop significantly after fine-tuning)")

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(results):
    """
    ROUGE measures word overlap between model output and reference answer.
    ROUGE-1: single word overlap
    ROUGE-2: two-word phrase overlap
    ROUGE-L: longest matching sequence
    All scores are 0 to 1. Higher is better.
    """
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=True
    )

    r1, r2, rl = [], [], []

    for item in results:
        scores = scorer.score(item["reference"], item["output"])
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    return {
        "ROUGE-1": round(np.mean(r1), 4),
        "ROUGE-2": round(np.mean(r2), 4),
        "ROUGE-L": round(np.mean(rl), 4),
    }

rouge_scores = compute_rouge(base_results)
print("ROUGE SCORES (base model)")
for metric, score in rouge_scores.items():
    print(f"  {metric}: {score}")
print("\n  (expect these to increase after fine-tuning)")

In [ ]:
def compute_repetition_rate(results, ngram_size=4):
    """
    Measures what % of output is repeated phrases.
    The loop you saw earlier would score nearly 100% here.
    A good answer should score close to 0%.
    """
    rates = []

    for item in results:
        words = item["output"].split()
        if len(words) < ngram_size:
            rates.append(0.0)
            continue

        ngrams = [
            tuple(words[i:i+ngram_size])
            for i in range(len(words) - ngram_size + 1)
        ]
        unique  = len(set(ngrams))
        total   = len(ngrams)
        repeat_rate = 1 - (unique / total) if total > 0 else 0
        rates.append(repeat_rate)

    return {"repetition_rate": round(np.mean(rates), 4)}

rep_scores = compute_repetition_rate(base_results)
print("REPETITION RATE (base model)")
print(f"  Rate: {rep_scores['repetition_rate']} (0=no repetition, 1=all repeated)")

In [ ]:
# This is the most important one for your professor.
# You manually score each output 1-5 on four dimensions.
# Do this yourself honestly — it takes about 10 minutes.

def human_eval_template(results):
    """
    Prints each question and output so you can score them.
    Fill in your scores in the scores dict below.
    """
    for i, item in enumerate(results):
        print(f"\n{'='*60}")
        print(f"Q{i+1}: {item['question']}")
        print(f"\nREFERENCE: {item['reference']}")
        print(f"\nMODEL OUTPUT: {item['output']}")
        print(f"\nScore on:")
        print(f"  Relevance  (1-5): did it address the question?")
        print(f"  Correctness(1-5): is the content factually right?")
        print(f"  Coherence  (1-5): does it make logical sense?")
        print(f"  Fluency    (1-5): does it read naturally?")

human_eval_template(base_results)

# After scoring, enter your scores here:
human_scores_base = {
    "Q1":  {"relevance": 1, "correctness": 1, "coherence": 1, "fluency": 2},
    "Q2":  {"relevance": 1, "correctness": 1, "coherence": 1, "fluency": 2},
    "Q3":  {"relevance": 2, "correctness": 1, "coherence": 1, "fluency": 2},
    "Q4":  {"relevance": 1, "correctness": 1, "coherence": 1, "fluency": 1},
    "Q5":  {"relevance": 1, "correctness": 1, "coherence": 2, "fluency": 2},
    # add all your questions...
}

# Compute averages
import pandas as pd
df_human = pd.DataFrame(human_scores_base).T
print("\nHuman evaluation averages (base model):")
print(df_human.mean().round(2))

In [ ]:
# Run this cell after all three stages to build the full comparison table.
# For now it just shows the base model row.

summary = {
    "Stage": [
        "1. Base TinyLlama",
        "2. Fine-tuned",        # fill in after fine-tuning
        "3. Fine-tuned + RAG",  # fill in after RAG
    ],
    "Perplexity (lower=better)": [
        perp_scores["mean_perplexity"], "—", "—"
    ],
    "ROUGE-L (higher=better)": [
        rouge_scores["ROUGE-L"], "—", "—"
    ],
    "Repetition rate (lower=better)": [
        rep_scores["repetition_rate"], "—", "—"
    ],
    "Human score /5 (higher=better)": [
        round(df_human.mean().mean(), 2), "—", "—"
    ],
}

df_summary = pd.DataFrame(summary)
print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)
print(df_summary.to_string(index=False))